<div style="background:#F4F4F3;border-top:4px solid #CEEB13;padding:30px 24px;text-align:center;">
<div style="font-family:'Roboto Mono',monospace;color:#395761;font-size:12px;letter-spacing:3px;">CRUSOE</div>
<div style="font-family:'Roboto Mono',monospace;color:#000000;font-size:32px;margin-top:12px;">How the API works</div>
<div style="font-family:system-ui,sans-serif;color:#4A4A4A;font-size:14.5px;margin-top:10px;">Ten minutes. Then open the chatbot and you will know exactly what it is doing.</div>
</div>

<img src="assets/how-it-works.svg" alt="Your question, the chat so far and any dropped files become one list of messages, which goes to Crusoe and streams back an answer that is saved to a JSON file." width="100%">

Run each cell with **Shift + Return**.

<div style="background:#F4F4F3;border-left:4px solid #CEEB13;padding:16px 20px;margin:26px 0 10px 0;"><div style="font-family:'Roboto Mono',monospace;color:#395761;font-size:11px;letter-spacing:2.5px;">STEP 1</div><div style="font-family:'Roboto Mono',monospace;color:#000000;font-size:20px;margin-top:6px;">The client</div>
<div style="font-family:system-ui,sans-serif;color:#4A4A4A;font-size:13.5px;margin-top:8px;">Crusoe speaks the OpenAI protocol. The only Crusoe-specific things are the address and your key.</div>
</div>

In [ ]:
from openai import OpenAI

from crusoe_chat import config

# Asks once if you have not set a key yet, then saves it to .env
api_key = config.ensure_api_key()

crusoe = OpenAI(
    api_key=api_key,
    base_url="https://api.inference.crusoecloud.com/v1",   # <- the only difference
)

print("Ready. Default model:", config.DEFAULT_MODEL)

<div style="background:#F4F4F3;border-left:4px solid #CEEB13;padding:16px 20px;margin:26px 0 10px 0;"><div style="font-family:'Roboto Mono',monospace;color:#395761;font-size:11px;letter-spacing:2.5px;">STEP 2</div><div style="font-family:'Roboto Mono',monospace;color:#000000;font-size:20px;margin-top:6px;">Your first call</div>
<div style="font-family:system-ui,sans-serif;color:#4A4A4A;font-size:13.5px;margin-top:8px;">One request, one answer.</div>
</div>

In [ ]:
response = crusoe.chat.completions.create(
    model=config.DEFAULT_MODEL,
    messages=[{"role": "user", "content": "Explain what an API key is, in two sentences."}],
)

print(response.choices[0].message.content)

Worth seeing what came back besides the text:

In [ ]:
print("model  ", response.model)
print("tokens ", response.usage.prompt_tokens, "in,", response.usage.completion_tokens, "out")
print("stop   ", response.choices[0].finish_reason)

<div style="background:#F4F4F3;border-left:4px solid #CEEB13;padding:16px 20px;margin:26px 0 10px 0;"><div style="font-family:'Roboto Mono',monospace;color:#395761;font-size:11px;letter-spacing:2.5px;">STEP 3</div><div style="font-family:'Roboto Mono',monospace;color:#000000;font-size:20px;margin-top:6px;">Streaming</div>
<div style="font-family:system-ui,sans-serif;color:#4A4A4A;font-size:13.5px;margin-top:8px;">The same call, printed as it arrives. This is why the chatbot feels quick.</div>
</div>

In [ ]:
stream = crusoe.chat.completions.create(
    model=config.DEFAULT_MODEL,
    messages=[{"role": "user",
               "content": "Name three uses for a large language model. One line each."}],
    stream=True,                                   # <- the only change
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="")

<div style="background:#F4F4F3;border-left:4px solid #CEEB13;padding:16px 20px;margin:26px 0 10px 0;"><div style="font-family:'Roboto Mono',monospace;color:#395761;font-size:11px;letter-spacing:2.5px;">STEP 4</div><div style="font-family:'Roboto Mono',monospace;color:#000000;font-size:20px;margin-top:6px;">Which models you can call</div>
<div style="font-family:system-ui,sans-serif;color:#4A4A4A;font-size:13.5px;margin-top:8px;">Switching model is a one-word change.</div>
</div>

In [ ]:
for name in sorted(m.id for m in crusoe.models.list().data):
    print(" ", name)

<div style="background:#F4F4F3;border-left:4px solid #CEEB13;padding:16px 20px;margin:26px 0 10px 0;"><div style="font-family:'Roboto Mono',monospace;color:#395761;font-size:11px;letter-spacing:2.5px;">STEP 5</div><div style="font-family:'Roboto Mono',monospace;color:#000000;font-size:20px;margin-top:6px;">Memory: the part everyone asks about</div>
<div style="font-family:system-ui,sans-serif;color:#4A4A4A;font-size:13.5px;margin-top:8px;">The API is stateless. It remembers nothing between calls. Watch it fail first.</div>
</div>

In [ ]:
# Two separate calls. The second has no idea the first happened.
crusoe.chat.completions.create(
    model=config.DEFAULT_MODEL,
    messages=[{"role": "user", "content": "My favourite colour is teal."}],
)

forgetful = crusoe.chat.completions.create(
    model=config.DEFAULT_MODEL,
    messages=[{"role": "user", "content": "What is my favourite colour?"}],
)

print(forgetful.choices[0].message.content)

It cannot know. Nothing was stored anywhere.

**The fix is not clever.** You send the earlier turns back with the new question. That is all memory is.

In [ ]:
conversation = [
    {"role": "user",      "content": "My favourite colour is teal."},
    {"role": "assistant", "content": "Teal it is."},
    {"role": "user",      "content": "What is my favourite colour?"},   # the new question
]

remembering = crusoe.chat.completions.create(
    model=config.DEFAULT_MODEL,
    messages=conversation,
)

print(remembering.choices[0].message.content)

That is the whole mechanism, and it has a cost: you re-send the conversation every turn, so a long chat uses more tokens than a short one.

The app keeps that in check by dropping the oldest turns once the history gets big:

In [ ]:
from crusoe_chat import memory

long_chat = [{"role": "user", "content": "x" * 500} for _ in range(20)]
print("before:", len(long_chat), "turns")

kept = memory.fit_to_budget(long_chat, budget_chars=2000)
print("after :", len(kept), "turns kept, the most recent ones")

<div style="background:#F4F4F3;border-left:4px solid #CEEB13;padding:16px 20px;margin:26px 0 10px 0;"><div style="font-family:'Roboto Mono',monospace;color:#395761;font-size:11px;letter-spacing:2.5px;">STEP 6</div><div style="font-family:'Roboto Mono',monospace;color:#000000;font-size:20px;margin-top:6px;">The same thing with LangChain</div>
<div style="font-family:system-ui,sans-serif;color:#4A4A4A;font-size:13.5px;margin-top:8px;">Optional. A framework can manage the message list for you, but it is the same list.</div>
</div>

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=config.DEFAULT_MODEL,
    api_key=api_key,
    base_url="https://api.inference.crusoecloud.com/v1",   # <- same address
)

chat_memory = [HumanMessage("My favourite colour is teal.")]
chat_memory.append(AIMessage(llm.invoke(chat_memory).content))
chat_memory.append(HumanMessage("What is my favourite colour? One word."))

print(llm.invoke(chat_memory).content)

`HumanMessage` and `AIMessage` are the same `{"role": ..., "content": ...}` dictionaries wearing a coat.

**The chatbot uses plain Python, not LangChain**, because at this size the list is easier to read than the framework around it. Reach for a framework when you need tools, agents or retrieval.

<div style="background:#F4F4F3;border-left:4px solid #CEEB13;padding:16px 20px;margin:26px 0 10px 0;"><div style="font-family:'Roboto Mono',monospace;color:#395761;font-size:11px;letter-spacing:2.5px;">STEP 7</div><div style="font-family:'Roboto Mono',monospace;color:#000000;font-size:20px;margin-top:6px;">Documents</div>
<div style="font-family:system-ui,sans-serif;color:#4A4A4A;font-size:13.5px;margin-top:8px;">No database, no embeddings. The file becomes text, and the text goes in the system message.</div>
</div>

In [ ]:
from crusoe_chat import documents

text = documents.read("README.md")
print(f"read {len(text):,} characters\n")

answer = crusoe.chat.completions.create(
    model=config.DEFAULT_MODEL,
    messages=[
        {"role": "system", "content": f"Answer using this document:\n\n{text}"},
        {"role": "user",   "content": "What does this project do? One sentence."},
    ],
)

print(answer.choices[0].message.content)

<div style="background:#F4F4F3;border-left:4px solid #CEEB13;padding:16px 20px;margin:26px 0 10px 0;"><div style="font-family:'Roboto Mono',monospace;color:#395761;font-size:11px;letter-spacing:2.5px;">STEP 8</div><div style="font-family:'Roboto Mono',monospace;color:#000000;font-size:20px;margin-top:6px;">That is the whole chatbot</div>
<div style="font-family:system-ui,sans-serif;color:#4A4A4A;font-size:13.5px;margin-top:8px;">Every piece you just ran, in five short files.</div>
</div>

| File | What you saw |
| --- | --- |
| `crusoe_chat/config.py` | Step 1, the key and the address |
| `crusoe_chat/client.py` | Steps 2 to 4, calling and streaming |
| `crusoe_chat/memory.py` | Step 5, building the message list and keeping it to a size |
| `crusoe_chat/documents.py` | Step 7, turning a file into text |
| `crusoe_chat/sessions.py` | Saving each conversation to `sessions/` as JSON |
| `app.py` | The browser interface, and nothing else |

Open any of them. None is longer than a page.

---

<div style="background:#F4F4F3;border-left:4px solid #CEEB13;padding:18px 22px;margin-top:22px;">
<div style="font-family:'Roboto Mono',monospace;color:#000000;font-size:18px;">Now run the chatbot</div>
<div style="font-family:system-ui,sans-serif;color:#4A4A4A;font-size:13.5px;margin-top:10px;">
In Terminal, from this folder:
<pre style="background:#FFFFFF;border:1px solid #E4E2E0;padding:10px;margin-top:8px;">./chat.sh</pre>
Pick a model, ask it something, drag in a document. Come back to an earlier chat from the list on the left. It is yours to keep.
</div>
</div>